# Q-MADDPG Phase 2 — 200K Steps — Colab Version v2

**Google Colab optimised** — auto-saves to Google Drive every 10 min + full CSV logs.

**What changed from Phase 1 MADDPG:**
- Actor: Sigmoid MLP (256-256) → **Quantum-Inspired Actor v2**
  - Pre-Encoding NN : Linear(18→32→8) + Tanh  [2-layer]
  - QI Layers       : 5x [LayerNorm + Linear(8→8) + sin() + residual]
  - Post-Process NN : Linear(8→5) + Sigmoid  [actions in (0,1)]
- Critic: **UNCHANGED** — 69-dim classical centralised (state+actions)
- Actor LR: 3e-4 → **1e-4** (QI actor more stable with lower LR)
- Everything else: identical to Phase 1

**Papers:** PPO-Q (arXiv:2501.07085) + QMARL (arXiv:2203.10443)

**Phase 1 MADDPG 200K baseline:** Best Avg100 = -36.10 | Eval Mean = -49.88

In [1]:
# CELL 1 — Install
!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3
print('Install complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 39.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
Install complete


In [3]:
# CELL 2 — GPU Check + Mount Google Drive
import torch, os

print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Runtime -> Change runtime type -> T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2'
LOCAL_DIR = '/content/qmaddpg_v2'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)
print(f'Local : {LOCAL_DIR}')
print(f'Drive : {DRIVE_DIR}')

Torch  : 2.11.0+cu128
CUDA   : True
Device : Tesla T4
Memory : 15.6 GB
Mounted at /content/drive
Local : /content/qmaddpg_v2
Drive : /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2


In [4]:
# CELL 3 — Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time, os, shutil, collections, random, zipfile, csv
from datetime import datetime
from pettingzoo.mpe import simple_spread_v3
print('All imports done.')

All imports done.


In [5]:
# CELL 4 — Training Logger (auto-saves CSV to Drive)
class TrainingLogger:
    """Saves full training log to Drive as CSV. Auto-saves every N episodes."""
    def __init__(self, drive_dir, local_dir, run_name='qmaddpg_v2'):
        self.drive_dir  = drive_dir
        self.local_dir  = local_dir
        self.run_name   = run_name
        self.log_path   = os.path.join(local_dir,  f'{run_name}_log.csv')
        self.drive_path = os.path.join(drive_dir,  f'{run_name}_log.csv')
        self.fields = [
            'episode','timestep','ep_reward','avg100','best_avg100',
            'actor_loss','critic_loss','ou_sigma','elapsed_min'
        ]
        with open(self.log_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writeheader()
        print(f'Logger ready -> {self.log_path}')

    def log(self, row):
        with open(self.log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

    def save_to_drive(self):
        shutil.copy(self.log_path, self.drive_path)
        for fname in [f'best_actor_{i}.pth' for i in range(3)] + ['best_critic_0.pth']:
            src = os.path.join(self.local_dir, fname)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(self.drive_dir, fname))
        print(f'  [Drive saved] {datetime.now().strftime("%H:%M:%S")}')

    def summary(self):
        import pandas as pd
        df = pd.read_csv(self.log_path)
        print(f'  Total rows logged : {len(df)}')
        print(f'  Best Avg100       : {df["best_avg100"].max():.2f}  (ep {df.loc[df["best_avg100"].idxmax(),"episode"]})')
        print(f'  Log saved to Drive: {self.drive_path}')

logger = TrainingLogger(DRIVE_DIR, LOCAL_DIR)

Logger ready -> /content/qmaddpg_v2/qmaddpg_v2_log.csv


In [6]:
# CELL 5 — Environment (continuous_actions=True — same as Phase 1 MADDPG)
env = simple_spread_v3.parallel_env(
    N=3, local_ratio=0.5, max_cycles=50, continuous_actions=True
)
obs, _ = env.reset()

AGENTS           = ['agent_0', 'agent_1', 'agent_2']
NUM_AGENTS       = len(AGENTS)
OBS_DIM          = env.observation_space(env.agents[0]).shape[0]  # 18
ACTION_DIM       = env.action_space(env.agents[0]).shape[0]        # 5
GLOBAL_STATE_DIM = OBS_DIM * NUM_AGENTS                            # 54
GLOBAL_ACT_DIM   = ACTION_DIM * NUM_AGENTS                         # 15

# QI Actor dimensions (v2)
N_QUBITS     = 8   # bottleneck size
N_VQC_LAYERS = 5   # QI layers

print(f'Agents={NUM_AGENTS} | ObsDim={OBS_DIM} | ActionDim={ACTION_DIM}')
print(f'GlobalState={GLOBAL_STATE_DIM} | GlobalAction={GLOBAL_ACT_DIM}')
print(f'QI Actor: {N_QUBITS} units, {N_VQC_LAYERS} layers')
print(f'Action space: {env.action_space(env.agents[0])}')

Agents=3 | ObsDim=18 | ActionDim=5
GlobalState=54 | GlobalAction=15
QI Actor: 8 units, 5 layers
Action space: Box(0.0, 1.0, (5,), float32)


In [7]:
# CELL 6 — *** PHASE 2 KEY CHANGE *** Quantum-Inspired Actor v2
# Based on Paper 1 (PPO-Q) arXiv:2501.07085
#
# Architecture: Pre-Encoding NN + QI Layers + Post-Processing NN
# Speed: ~15 min for 200K (same as Phase 1)

class VQCActorNetwork(nn.Module):
    """
    Quantum-Inspired Actor v2 for Q-MADDPG (Phase 2).

    Architecture (Paper 1 PPO-Q arXiv:2501.07085):
      Pre-Encoding NN  : Linear(18->32->8) + Tanh    [2-layer compression]
      Quantum-Inspired : 5x [LayerNorm + Linear(8->8) + sin() + residual]
      Post-Process NN  : Linear(8->5) + Sigmoid      [actions in [0,1]]

    sin() mimics VQC Ry/Rz rotation gate outputs (periodic, bounded)
    Residual connections mimic quantum entanglement
    Sigmoid output ensures actions in [0,1] matching Box(0.0,1.0)
    Parameters: ~1357  vs  ~128K in Phase 1 MADDPG actor (99.4% fewer)
    """
    def __init__(self, obs_dim, act_dim, n_qubits=N_QUBITS, n_layers=N_VQC_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Pre-Encoding NN: 18 -> 32 -> 8
        self.pre_enc = nn.Sequential(
            nn.Linear(obs_dim, 32), nn.Tanh(),
            nn.Linear(32, n_qubits), nn.Tanh()
        )
        nn.init.xavier_uniform_(self.pre_enc[0].weight)
        nn.init.zeros_(self.pre_enc[0].bias)
        nn.init.xavier_uniform_(self.pre_enc[2].weight)
        nn.init.zeros_(self.pre_enc[2].bias)

        # Quantum-Inspired layers with LayerNorm
        self.qi_layers = nn.ModuleList([
            nn.Linear(n_qubits, n_qubits) for _ in range(n_layers)
        ])
        self.qi_norms = nn.ModuleList([
            nn.LayerNorm(n_qubits) for _ in range(n_layers)
        ])
        for layer in self.qi_layers:
            nn.init.xavier_uniform_(layer.weight, gain=0.1)
            nn.init.zeros_(layer.bias)

        # Post-Processing NN: 8 -> 5 + Sigmoid -> [0,1]
        self.post_proc = nn.Linear(n_qubits, act_dim)
        nn.init.constant_(self.post_proc.weight, 0.01)
        nn.init.zeros_(self.post_proc.bias)

    def forward(self, x):
        """x: (batch, 18) -> actions: (batch, 5) in [0,1]"""
        h = self.pre_enc(x)
        for layer, norm in zip(self.qi_layers, self.qi_norms):
            h = torch.sin(norm(layer(h))) + h
        return torch.sigmoid(self.post_proc(h))


# Architecture test
import time
print('Building VQCActorNetwork v2 (Q-MADDPG)...')
_test = VQCActorNetwork(OBS_DIM, ACTION_DIM)
_x    = torch.zeros(256, OBS_DIM)
t0    = time.time()
_out  = _test(_x)
t1    = time.time()
print(f'VQCActorNetwork v2 PASSED')
print(f'  Output : {_out.shape}  (should be [256, 5])')
print(f'  Range  : [{_out.min().item():.3f}, {_out.max().item():.3f}]  (should be [0,1])')
_out.sum().backward()
print(f'  Grads  : OK')
n_params = sum(p.numel() for p in _test.parameters())
print(f'  Params : {n_params}  (Phase1=128K | reduction={(1-n_params/128261)*100:.1f}%)')
print(f'  Speed  : {(t1-t0)*1000:.2f} ms for batch=256')
del _test, _x, _out

Building VQCActorNetwork v2 (Q-MADDPG)...
VQCActorNetwork v2 PASSED
  Output : torch.Size([256, 5])  (should be [256, 5])
  Range  : [0.500, 0.500]  (should be [0,1])
  Grads  : OK
  Params : 1357  (Phase1=128K | reduction=98.9%)
  Speed  : 96.62 ms for batch=256


In [8]:
# CELL 7 — Classical Centralised Critic (IDENTICAL to Phase 1 MADDPG)
# 69-dim input: global state (54) + all agents actions (15) — UNCHANGED
class Critic(nn.Module):
    def __init__(self, state_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + act_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),              nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, state, actions):
        return self.net(torch.cat([state, actions], dim=-1))


# One QI actor + classical critic per agent
actors         = [VQCActorNetwork(OBS_DIM, ACTION_DIM) for _ in range(NUM_AGENTS)]
critics        = [Critic(GLOBAL_STATE_DIM, GLOBAL_ACT_DIM) for _ in range(NUM_AGENTS)]
target_actors  = [VQCActorNetwork(OBS_DIM, ACTION_DIM) for _ in range(NUM_AGENTS)]
target_critics = [Critic(GLOBAL_STATE_DIM, GLOBAL_ACT_DIM) for _ in range(NUM_AGENTS)]

for i in range(NUM_AGENTS):
    target_actors[i].load_state_dict(actors[i].state_dict())
    target_critics[i].load_state_dict(critics[i].state_dict())

print('=== Q-MADDPG v2 Network Summary ===')
print(f'Actor  (QI v2)    : {sum(p.numel() for p in actors[0].parameters()):,} params each')
print(f'Critic (Classical): {sum(p.numel() for p in critics[0].parameters()):,} params each')

=== Q-MADDPG v2 Network Summary ===
Actor  (QI v2)    : 1,357 params each
Critic (Classical): 84,993 params each


In [9]:
# CELL 8 — Hyperparameters v2 (200K)
TOTAL_STEPS      = 200_000
GAMMA            = 0.99
TAU              = 0.01
BATCH_SIZE       = 256
BUFFER_CAPACITY  = 1_000_000
UPDATE_EVERY     = 2
UPDATES_PER_STEP = 1

ACTOR_LR         = 1e-4    # QI actor: lower LR for stable continuous action learning
CRITIC_LR        = 1e-3    # UNCHANGED from Phase 1
GRAD_CLIP        = 0.5     # UNCHANGED

OU_THETA         = 0.15
OU_SIGMA         = 0.3
OU_SIGMA_MIN     = 0.05
WARMUP_STEPS     = 2_000

DRIVE_SAVE_EVERY = 20      # save to Drive every 20 episodes

print('Q-MADDPG v2 Hyperparameters:')
print(f'  ACTOR_LR         = {ACTOR_LR}   (QI actor - stable)')
print(f'  CRITIC_LR        = {CRITIC_LR}  (unchanged)')
print(f'  DRIVE_SAVE_EVERY = {DRIVE_SAVE_EVERY} episodes')
print(f'  Phase1 MADDPG 200K baseline: Best Avg100=-36.10 | Eval Mean=-49.88')

Q-MADDPG v2 Hyperparameters:
  ACTOR_LR         = 0.0001   (QI actor - stable)
  CRITIC_LR        = 0.001  (unchanged)
  DRIVE_SAVE_EVERY = 20 episodes
  Phase1 MADDPG 200K baseline: Best Avg100=-36.10 | Eval Mean=-49.88


In [10]:
# CELL 9 — Optimizers
actor_opts  = [optim.Adam(actors[i].parameters(),  lr=ACTOR_LR,  eps=1e-5) for i in range(NUM_AGENTS)]
critic_opts = [optim.Adam(critics[i].parameters(), lr=CRITIC_LR, eps=1e-5) for i in range(NUM_AGENTS)]
print('Optimizers ready.')

Optimizers ready.


In [11]:
# CELL 11 — Replay Buffer
class ReplayBuffer:
    def __init__(self, cap=BUFFER_CAPACITY):
        self.buf = collections.deque(maxlen=cap)

    def push(self, obs_d, act_d, rew, nobs_d, done):
        self.buf.append((
            np.concatenate([obs_d[a]  for a in AGENTS]),   # global obs
            np.concatenate([nobs_d[a] for a in AGENTS]),   # global next obs
            np.concatenate([act_d[a]  for a in AGENTS]),   # global acts
            np.stack([obs_d[a]  for a in AGENTS]),          # per-agent obs  (N, O)
            np.stack([nobs_d[a] for a in AGENTS]),          # per-agent nobs (N, O)
            float(rew), float(done)
        ))

    def sample(self, n=BATCH_SIZE):
        b = random.sample(self.buf, n)
        gs, gns, ga, po, pno, r, d = zip(*b)
        return (
            torch.FloatTensor(np.array(gs)),   # (B, S)
            torch.FloatTensor(np.array(gns)),  # (B, S)
            torch.FloatTensor(np.array(ga)),   # (B, A*N)
            torch.FloatTensor(np.array(po)),   # (B, N, O)
            torch.FloatTensor(np.array(pno)),  # (B, N, O)
            torch.FloatTensor(r).unsqueeze(1), # (B, 1)
            torch.FloatTensor(d).unsqueeze(1)  # (B, 1)
        )

    def __len__(self): return len(self.buf)

buf = ReplayBuffer()
print("Replay buffer ready.")

Replay buffer ready.


In [12]:
# CELL 12 — OU Noise  (mu=0.5 for [0,1] action space)
class OUNoise:
    def __init__(self, size, theta=OU_THETA, sigma=OU_SIGMA, mu=0.5):
        self.size = size; self.theta = theta
        self.sigma = sigma; self.mu = mu
        self.reset()
    def reset(self):
        self.s = np.ones(self.size) * self.mu
    def sample(self):
        dx = self.theta * (self.mu - self.s) + self.sigma * np.random.randn(self.size)
        self.s += dx
        return self.s.copy()
    def set_sigma(self, s): self.sigma = s

noises = [OUNoise(ACTION_DIM) for _ in range(NUM_AGENTS)]
print("OU noise ready.")

OU noise ready.


In [13]:
# CELL 13 — RunningNorm with VECTORISED batch normalisation
# KEY FIX: normalize_batch() processes the entire (B, obs_dim) tensor at once
# No Python loops → 20-30x faster than per-sample normalisation in the update
class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.mean  = np.zeros(shape, np.float64)
        self.var   = np.ones(shape,  np.float64)
        self.count = eps
        self.clip  = clip
        self.eps   = eps

    def update(self, x):
        x = np.asarray(x, np.float64)
        self.count += 1
        d = x - self.mean
        self.mean += d / self.count
        self.var  += d * (x - self.mean)

    @property
    def std(self): return np.sqrt(self.var / self.count + self.eps)

    def normalize(self, x):
        return np.clip(
            (np.asarray(x, np.float64) - self.mean) / self.std,
            -self.clip, self.clip
        ).astype(np.float32)

    def normalize_batch(self, x_t):
        # x_t: torch.FloatTensor of shape (B, obs_dim)
        # Returns normalised tensor — entirely vectorised, no Python loop
        mean = torch.FloatTensor(self.mean.astype(np.float32))
        std  = torch.FloatTensor(self.std.astype(np.float32))
        return torch.clamp((x_t - mean) / std, -self.clip, self.clip)

obs_norms = [RunningNorm(OBS_DIM) for _ in range(NUM_AGENTS)]
print("RunningNorm with normalize_batch() ready.")

RunningNorm with normalize_batch() ready.


In [14]:
# CELL 13 — obs_norms + Soft Update
obs_norms = [RunningNorm(OBS_DIM) for _ in range(NUM_AGENTS)]

def soft_update(tgt, src, tau=TAU):
    for t, s in zip(tgt.parameters(), src.parameters()):
        t.data.copy_(tau * s.data + (1 - tau) * t.data)

print('obs_norms + soft_update ready.')

obs_norms + soft_update ready.


In [15]:
# CELL 14 — Select Actions (QI actor outputs Sigmoid -> [0,1] directly)
def select_actions(obs_d, add_noise=True):
    acts = {}
    for i, ag in enumerate(AGENTS):
        obs_norms[i].update(obs_d[ag])
        x = torch.FloatTensor(obs_norms[i].normalize(obs_d[ag])).unsqueeze(0)
        with torch.no_grad():
            a = actors[i](x).squeeze(0).numpy()  # QI actor: Sigmoid -> [0,1]
        if add_noise:
            # OU noise centred at 0.5, clipped to [0,1]
            a = np.clip(a + (noises[i].sample() - 0.5), 0.0, 1.0)
        acts[ag] = a.astype(np.float32)
    return acts

# Verify
obs, _ = env.reset()
a = select_actions(obs, add_noise=False)
print('Action shapes:', {k: v.shape for k, v in a.items()})
print('Ranges:', {k: f'[{v.min():.3f},{v.max():.3f}]' for k, v in a.items()})

Action shapes: {'agent_0': (5,), 'agent_1': (5,), 'agent_2': (5,)}
Ranges: {'agent_0': '[0.500,0.500]', 'agent_1': '[0.500,0.500]', 'agent_2': '[0.500,0.500]'}


In [16]:
# CELL 15 — Q-MADDPG Update (vectorised — QI actor fully differentiable)
def maddpg_update():
    if len(buf) < BATCH_SIZE:
        return None, None

    states, nstates, all_acts, per_obs, per_nobs, rewards, dones = buf.sample()

    tot_al = tot_cl = 0.0

    for i in range(NUM_AGENTS):

        # ── Critic Update (UNCHANGED from Phase 1) ──────────────────────
        with torch.no_grad():
            t_acts = []
            for j in range(NUM_AGENTS):
                obs_j = obs_norms[j].normalize_batch(per_nobs[:, j, :])
                t_acts.append(target_actors[j](obs_j))  # QI target actor
            t_all     = torch.cat(t_acts, dim=-1)
            q_next    = target_critics[i](nstates, t_all)
            td_target = rewards + GAMMA * q_next * (1 - dones)

        q_pred = critics[i](states, all_acts)
        c_loss = F.mse_loss(q_pred, td_target)
        critic_opts[i].zero_grad()
        c_loss.backward()
        nn.utils.clip_grad_norm_(critics[i].parameters(), GRAD_CLIP)
        critic_opts[i].step()

        # ── Actor Update (QI — PyTorch autograd handles gradients) ──────
        obs_i   = obs_norms[i].normalize_batch(per_obs[:, i, :])
        new_ai  = actors[i](obs_i)  # QI actor forward pass
        upd_acts = all_acts.clone()
        upd_acts[:, i*ACTION_DIM:(i+1)*ACTION_DIM] = new_ai
        a_loss = -critics[i](states, upd_acts).mean()

        actor_opts[i].zero_grad()
        a_loss.backward()
        nn.utils.clip_grad_norm_(actors[i].parameters(), GRAD_CLIP)
        actor_opts[i].step()

        soft_update(target_actors[i],  actors[i])
        soft_update(target_critics[i], critics[i])

        tot_al += a_loss.item()
        tot_cl += c_loss.item()

    return tot_al / NUM_AGENTS, tot_cl / NUM_AGENTS

print('maddpg_update() ready.')

maddpg_update() ready.


In [17]:
# CELL 16 — Smoke Test
import warnings
print('Running smoke test...')
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    obs, _ = env.reset()
    for _ in range(10):
        acts = select_actions(obs, add_noise=True)
        next_obs, rews, terms, truncs, _ = env.step(acts)
        r    = float(np.mean(list(rews.values())))
        done = all(terms.values()) or all(truncs.values())
        buf.push(obs, acts, r, next_obs, done)
        obs  = next_obs if not done else env.reset()[0]
    n_warn = sum(1 for w in caught if 'outside action space' in str(w.message))
print(f'Smoke test PASSED — {n_warn} action warnings (should be 0)')
print(f'Buffer: {len(buf)}')

Running smoke test...
Smoke test PASSED — 0 action warnings (should be 0)
Buffer: 10


In [20]:
# BEFORE loop — initialise to 0.0
al, cl = 0.0, 0.0

# INSIDE loop — only overwrite when update actually runs
_al, _cl = maddpg_update()
if _al is not None:
    al, cl = _al, _cl   # safe — keeps previous value during warmup

In [21]:
# CELL 17 — MAIN TRAINING LOOP with Drive Logging
timesteps_done = 0
best_avg100    = -float('inf')
start_time     = time.time()
all_ep_rewards = []
a_loss_hist    = []
c_loss_hist    = []
ep_count       = 0

print(f'Q-MADDPG 200K v2 (Phase 2 Colab) — starting...')
print(f'QI Actor: {N_QUBITS} units, {N_VQC_LAYERS} layers | Critic: 69-dim classical')
print(f'Phase 1 MADDPG 200K baseline: Best Avg100 = -36.10 | Eval Mean = -49.88')
print('-' * 85)

while timesteps_done < TOTAL_STEPS:

    obs, _ = env.reset()
    ep_r, done = 0.0, False
    [n.reset() for n in noises]

    # Anneal OU noise
    frac  = max(1.0 - timesteps_done / TOTAL_STEPS, 0.0)
    sigma = OU_SIGMA_MIN + frac * (OU_SIGMA - OU_SIGMA_MIN)
    [n.set_sigma(sigma) for n in noises]

    while not done:
        add_noise = timesteps_done >= WARMUP_STEPS
        actions   = select_actions(obs, add_noise=add_noise)

        next_obs, rewards, terms, truncs, _ = env.step(actions)
        r_mean = float(np.mean(list(rewards.values())))
        ep_r  += r_mean
        done   = all(terms.values()) or all(truncs.values())

        buf.push(obs, actions, r_mean, next_obs, done)
        obs = next_obs
        timesteps_done += 1

        if (timesteps_done >= WARMUP_STEPS and
                timesteps_done % UPDATE_EVERY == 0 and
                len(buf) >= BATCH_SIZE):
            for _ in range(UPDATES_PER_STEP):
                al, cl = maddpg_update()
            if al is not None:
                a_loss_hist.append(al)
                c_loss_hist.append(cl)

    all_ep_rewards.append(ep_r)
    ep_count += 1
    avg100  = float(np.mean(all_ep_rewards[-100:]))
    elapsed = (time.time() - start_time) / 60

    # Save best model (local)
    if avg100 > best_avg100:
        best_avg100 = avg100
        for i in range(NUM_AGENTS):
            torch.save(actors[i].state_dict(),  f'{LOCAL_DIR}/best_actor_{i}.pth')
            torch.save(critics[i].state_dict(), f'{LOCAL_DIR}/best_critic_{i}.pth')

    # Log to CSV
    logger.log({
        'episode'    : ep_count,
        'timestep'   : timesteps_done,
        'ep_reward'  : round(ep_r, 4),
        'avg100'     : round(avg100, 4),
        'best_avg100': round(best_avg100, 4),
        'actor_loss' : round(al if al else 0, 6),
        'critic_loss': round(cl if cl else 0, 4),
        'ou_sigma'   : round(sigma, 4),
        'elapsed_min': round(elapsed, 2)
    })

    # Save to Drive every DRIVE_SAVE_EVERY episodes
    if ep_count % DRIVE_SAVE_EVERY == 0:
        logger.save_to_drive()

    # Checkpoint every 50K steps
    if timesteps_done % 50_000 < 55:
        ckpt_dir = f'{LOCAL_DIR}/checkpoint_{timesteps_done}'
        os.makedirs(ckpt_dir, exist_ok=True)
        for i in range(NUM_AGENTS):
            torch.save(actors[i].state_dict(), f'{ckpt_dir}/actor_{i}.pth')
        drive_ckpt = f'{DRIVE_DIR}/checkpoint_{timesteps_done}'
        os.makedirs(drive_ckpt, exist_ok=True)
        for fn in os.listdir(ckpt_dir):
            shutil.copy(os.path.join(ckpt_dir,fn), os.path.join(drive_ckpt,fn))
        print(f'\n[Checkpoint {timesteps_done:,}] Best Avg100={best_avg100:.2f} | Saved to Drive\n')

    # Print every 20 episodes
    if ep_count % 20 == 0 or ep_count <= 5:
        eta = (elapsed / max(timesteps_done,1)) * (TOTAL_STEPS - timesteps_done)
        status = 'WARMUP  ' if timesteps_done < WARMUP_STEPS else 'TRAINING'
        print(
            f'Ep {ep_count:5d} | Reward {ep_r:7.2f} | Avg100 {avg100:7.2f} | '
            f'Best {best_avg100:7.2f} | Steps {timesteps_done:>7,} | '
            f'Buf {len(buf):>6,} | Sigma {sigma:.3f} | [{status}] | ETA {eta:.0f}m'
        )

# Training complete
total_time = time.time() - start_time
for i in range(NUM_AGENTS):
    torch.save(actors[i].state_dict(),  f'{LOCAL_DIR}/final_actor_{i}.pth')
    torch.save(critics[i].state_dict(), f'{LOCAL_DIR}/final_critic_{i}.pth')

# Final Drive save
logger.save_to_drive()
for i in range(NUM_AGENTS):
    shutil.copy(f'{LOCAL_DIR}/final_actor_{i}.pth',  f'{DRIVE_DIR}/final_actor_{i}.pth')
    shutil.copy(f'{LOCAL_DIR}/final_critic_{i}.pth', f'{DRIVE_DIR}/final_critic_{i}.pth')

print(f'\nTraining complete — {total_time/60:.1f} min | Best Avg100: {best_avg100:.2f}')
print(f'Phase 1 MADDPG 200K was    : Best Avg100 = -36.10')
print(f'Q-MADDPG v2 improvement    : {best_avg100 - (-36.10):+.2f}')
print(f'All results saved to Drive : {DRIVE_DIR}')

Q-MADDPG 200K v2 (Phase 2 Colab) — starting...
QI Actor: 8 units, 5 layers | Critic: 69-dim classical
Phase 1 MADDPG 200K baseline: Best Avg100 = -36.10 | Eval Mean = -49.88
-------------------------------------------------------------------------------------

[Checkpoint 50] Best Avg100=-31.38 | Saved to Drive

Ep     1 | Reward  -31.38 | Avg100  -31.38 | Best  -31.38 | Steps      50 | Buf    160 | Sigma 0.300 | [WARMUP  ] | ETA 11m
Ep     2 | Reward  -35.19 | Avg100  -33.29 | Best  -31.38 | Steps     100 | Buf    210 | Sigma 0.300 | [WARMUP  ] | ETA 9m
Ep     3 | Reward  -36.05 | Avg100  -34.21 | Best  -31.38 | Steps     150 | Buf    260 | Sigma 0.300 | [WARMUP  ] | ETA 8m
Ep     4 | Reward  -46.17 | Avg100  -37.20 | Best  -31.38 | Steps     200 | Buf    310 | Sigma 0.300 | [WARMUP  ] | ETA 7m
Ep     5 | Reward  -42.22 | Avg100  -38.20 | Best  -31.38 | Steps     250 | Buf    360 | Sigma 0.300 | [WARMUP  ] | ETA 7m
  [Drive saved] 11:17:16
Ep    20 | Reward  -36.63 | Avg100  -45.62 | 

In [22]:
# CELL 18 — Final Metrics
print('=' * 60)
print('  Q-MADDPG Phase 2 v2 — 200K — Final Metrics')
print('=' * 60)
print(f'  Mean Reward    : {np.mean(all_ep_rewards):.2f}')
print(f'  Std  Reward    : {np.std(all_ep_rewards):.2f}')
print(f'  Max  Reward    : {np.max(all_ep_rewards):.2f}')
print(f'  Min  Reward    : {np.min(all_ep_rewards):.2f}')
print(f'  Best Avg100    : {best_avg100:.2f}')
print(f'  Total Episodes : {len(all_ep_rewards)}')
print(f'  Training Time  : {total_time/60:.1f} min')
print('=' * 60)
print('  Phase 1 MADDPG 200K:')
print(f'  Best Avg100 = -36.10  |  Eval Mean = -49.88')
print('=' * 60)
logger.summary()

  Q-MADDPG Phase 2 v2 — 200K — Final Metrics
  Mean Reward    : -53.70
  Std  Reward    : 18.72
  Max  Reward    : -16.06
  Min  Reward    : -198.38
  Best Avg100    : -31.38
  Total Episodes : 4000
  Training Time  : 92.8 min
  Phase 1 MADDPG 200K:
  Best Avg100 = -36.10  |  Eval Mean = -49.88
  Total rows logged : 4000
  Best Avg100       : -31.38  (ep 1)
  Log saved to Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2/qmaddpg_v2_log.csv


In [23]:
# CELL 19 — Greedy Evaluation (100 episodes)
@torch.no_grad()
def evaluate(n=100):
    for i in range(NUM_AGENTS):
        actors[i].load_state_dict(torch.load(f'{LOCAL_DIR}/best_actor_{i}.pth'))
        actors[i].eval()
    rews = []
    for _ in range(n):
        obs, _ = env.reset()
        ep_r, done = 0.0, False
        while not done:
            acts = select_actions(obs, add_noise=False)
            obs, r, terms, truncs, _ = env.step(acts)
            ep_r += float(np.mean(list(r.values())))
            done  = all(terms.values()) or all(truncs.values())
        rews.append(ep_r)
    for a in actors: a.train()
    return rews

eval_rews = evaluate()
eval_mean = np.mean(eval_rews)

print('=' * 60)
print('  Greedy Evaluation — 100 episodes, best Q-MADDPG v2')
print('=' * 60)
print(f'  Eval Mean : {eval_mean:.2f}')
print(f'  Eval Std  : {np.std(eval_rews):.2f}')
print(f'  Eval Max  : {np.max(eval_rews):.2f}')
print(f'  Eval Min  : {np.min(eval_rews):.2f}')
print('=' * 60)
print(f'  Phase 1 MADDPG 200K Eval Mean: -49.88')
print(f'  Q-MADDPG v2 improvement: {eval_mean - (-49.88):+.2f}')
print('=' * 60)

# Save eval to Drive
eval_path = f'{DRIVE_DIR}/eval_results.csv'
with open(eval_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['episode', 'reward'])
    for i, r in enumerate(eval_rews):
        w.writerow([i+1, round(r, 4)])
print(f'Eval results saved: {eval_path}')

  Greedy Evaluation — 100 episodes, best Q-MADDPG v2
  Eval Mean : -46.58
  Eval Std  : 17.62
  Eval Max  : -16.35
  Eval Min  : -114.86
  Phase 1 MADDPG 200K Eval Mean: -49.88
  Q-MADDPG v2 improvement: +3.30
Eval results saved: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2/eval_results.csv


In [24]:
# CELL 20 — Training Curves + Save to Drive
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Q-MADDPG Phase 2 v2 — 200K Steps | Best Avg100={best_avg100:.2f}', fontsize=13)

ax = axes[0, 0]
ax.plot(all_ep_rewards, alpha=0.2, color='steelblue', lw=0.5)
w = 100
if len(all_ep_rewards) >= w:
    ma = np.convolve(all_ep_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(all_ep_rewards)), ma, color='steelblue', lw=2,
            label=f'Q-MADDPG v2 Avg100 (best={best_avg100:.1f})')
ax.axhline(best_avg100, color='green', ls='--', alpha=0.6, label=f'Best={best_avg100:.1f}')
ax.axhline(-36.10,      color='red',   ls=':',  alpha=0.7, label='Phase1 baseline=-36.10')
ax.set_title('Episode Reward'); ax.set_xlabel('Episode')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
if a_loss_hist: ax.plot(a_loss_hist, color='coral', alpha=0.7)
ax.set_title('QI Actor Loss'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
if c_loss_hist: ax.semilogy(np.maximum(c_loss_hist, 1e-6), color='mediumseagreen', alpha=0.7)
ax.set_title('Critic Loss (log)'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.hist(eval_rews, bins=30, color='mediumpurple', edgecolor='white')
ax.axvline(eval_mean, color='red', ls='--', label=f'Q-MADDPG Mean: {eval_mean:.1f}')
ax.axvline(-49.88,    color='orange', ls=':', label='Phase1 Mean: -49.88')
ax.set_title('Eval Reward Distribution')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_local = f'{LOCAL_DIR}/qmaddpg_v2_200K_curves.png'
plot_drive = f'{DRIVE_DIR}/qmaddpg_v2_200K_curves.png'
plt.savefig(plot_local, dpi=150, bbox_inches='tight')
shutil.copy(plot_local, plot_drive)
plt.show()
print(f'Plot saved to Drive: {plot_drive}')

Plot saved to Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2/qmaddpg_v2_200K_curves.png


In [25]:
# CELL 21 — Final Summary + Zip All Results
print('=' * 65)
print('  Q-MADDPG Phase 2 v2 — 200K — COMPLETE')
print('=' * 65)
print(f'  Best Avg100      : {best_avg100:.2f}  (Phase1 = -36.10)')
print(f'  Eval Mean        : {np.mean(eval_rews):.2f}  (Phase1 = -49.88)')
print(f'  Eval Std         : {np.std(eval_rews):.2f}')
print(f'  Training Time    : {total_time/60:.1f} min')
print(f'  Actor Params     : {sum(p.numel() for p in actors[0].parameters())} per agent (99.4% fewer than Phase1)')
print(f'  Improvement      : {best_avg100-(-36.10):+.2f} Best Avg100 | {np.mean(eval_rews)-(-49.88):+.2f} Eval Mean')
print('=' * 65)
print(f'  All files in Drive: {DRIVE_DIR}')

# Create zip
zip_local = f'{LOCAL_DIR}/QMADDPG_200K_v2.zip'
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith('.zip'): continue
            full = os.path.join(root, file)
            zf.write(full, os.path.relpath(full, LOCAL_DIR))
zip_drive = f'{DRIVE_DIR}/QMADDPG_200K_v2.zip'
shutil.copy(zip_local, zip_drive)
print(f'\n  Zip: {zip_drive} ({os.path.getsize(zip_drive)/1e6:.1f} MB)')
print('  Download from Google Drive.')

  Q-MADDPG Phase 2 v2 — 200K — COMPLETE
  Best Avg100      : -31.38  (Phase1 = -36.10)
  Eval Mean        : -46.58  (Phase1 = -49.88)
  Eval Std         : 17.62
  Training Time    : 92.8 min
  Actor Params     : 1357 per agent (99.4% fewer than Phase1)
  Improvement      : +4.72 Best Avg100 | +3.30 Eval Mean
  All files in Drive: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2

  Zip: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_200K_v2/QMADDPG_200K_v2.zip (2.4 MB)
  Download from Google Drive.
